In [3]:
import whisper
import sounddevice as sd
import numpy as np
import queue

In [4]:
model = whisper.load_model("tiny.en")

In [ ]:
SAMPLE_RATE = 16000
WINDOW = 2       # seconds of audio per transcription
STRIDE = 1       # slide forward by this many seconds

In [6]:
audio_buffer = np.empty(0, dtype=np.float32)
q = queue.Queue()

In [7]:
def callback(indata, frames, time, status):
    q.put(indata[:, 0].copy())

In [9]:
window_samples = SAMPLE_RATE*WINDOW

In [16]:
def transcribe_stream():
    global audio_buffer
    global prev_text
    window_samples = int(SAMPLE_RATE * WINDOW)
    stride_samples = int(SAMPLE_RATE * STRIDE)

    with sd.InputStream(samplerate=SAMPLE_RATE, channels=1,
                        dtype='float32', callback=callback):
        print("🎙 Listening...")
        while True:
            # Drain queue into buffer
            while not q.empty():
                audio_buffer = np.concatenate([audio_buffer, q.get()])

            # if len(audio_buffer) >= window_samples:
            #     chunk = audio_buffer[:window_samples]
            #     audio_buffer = audio_buffer[stride_samples:]  # slide window

            #     result = model.transcribe(chunk, fp16=False, language="en")
            #     text = result["text"].strip()
            #     if text:
            #         print(f">> {text}")
                    
            if len(audio_buffer) >= window_samples:
                chunk = audio_buffer[:window_samples]
                audio_buffer = audio_buffer[stride_samples:]  # slide window

                chunk = chunk.astype(np.float32)

                # Detect silence: if RMS energy is too low, skip transcription
                rms_energy = np.sqrt(np.mean(chunk ** 2))
                if rms_energy < 0.01:  # Adjust threshold based on your mic
                    continue  # Skip transcribing silence

                result = model.transcribe(chunk, fp16=False, language="en")
                text = result["text"].strip()

                if text:
                    # Incremental printing: extract only new text
                    if text.startswith(prev_text):
                        new_text = text[len(prev_text):].strip()
                    else:
                        new_text = text  # Fallback if transcription doesn't match previous

                    if new_text and len(new_text.split()) > 0:  # Only print if not empty
                        print(new_text, end=" ", flush=True)

                    prev_text = text  # Update for next chunk
transcribe_stream()

🎙 Listening...
Hello. How are you? I'll allow you. What? What are you doing? IUD. Fuck you. You are perfect. performing. forming better now. 

KeyboardInterrupt: 